
# Statistics for Machine Learning — **Exercises XP Gold** (Solution)



This notebook follows the XP Gold brief and completes the analyses with clean code and clear interpretation:

1. Load & prepare the **Students Performance** dataset  
2. Visualize distributions & group differences (matplotlib only)  
3. **Independent two-sample t-tests** (Male vs Female) for Math, Reading, Writing  
4. Assumptions: **normality** (Shapiro) & **homoscedasticity** (Levene)  
5. **One-Way ANOVA** across multi-level factors (e.g., race/ethnicity, lunch, test preparation)  
6. **Post-hoc tests** (Tukey HSD or Bonferroni fallback)  
7. Insights & conclusions

> Note: Stop when you see “**XP Ninja**” in this notebook, as requested.



## 1) Setup & Data Loading
- Load `Students Performance.csv` (allow for common file name variants)
- Standardize column names
- Quick inspection


In [ ]:

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import stats

# Try known filenames
candidates = [
    '/mnt/data/Students Performance.csv',
    '/mnt/data/StudentsPerformance.csv',
    'Students Performance.csv',
    'StudentsPerformance.csv'
]
path = None
for p in candidates:
    if os.path.exists(p):
        path = p
        break

if path is None:
    raise FileNotFoundError("Students Performance dataset not found in expected locations.")

df = pd.read_csv(path)

# Standardize columns (common variants)
df.columns = [c.strip().lower().replace(' ', '_') for c in df.columns]

# Attempt to align typical names
rename_map = {
    'math_score':'math',
    'reading_score':'reading',
    'writing_score':'writing',
    'math_score_':'math',
    'reading_score_':'reading',
    'writing_score_':'writing',
    'gender ':'gender'
}
df = df.rename(columns=rename_map)

print('Shape:', df.shape)
display(df.head())
print("\nColumns:", list(df.columns))

# Basic info
display(df.info())
display(df.describe(include='all').T.head(20))



## 2) Exploratory Data Analysis & Visualizations (Matplotlib Only)
- Missing values check
- Distributions: Math, Reading, Writing
- Group summaries by gender
- Boxplots by gender


In [ ]:

# Missing values
print('Missing values per column:')
print(df.isnull().sum())

# Determine target score columns
score_cols = [c for c in ['math','reading','writing'] if c in df.columns]
if len(score_cols) < 3:
    print("Warning: Not all expected score columns found. Detected:", score_cols)

# Histograms
for col in score_cols:
    if np.issubdtype(df[col].dtype, np.number):
        plt.figure(figsize=(6,4))
        plt.hist(df[col].dropna(), bins=25)
        plt.title(f'Distribution of {col.title()} Score')
        plt.xlabel(col.title())
        plt.ylabel('Count')
        plt.tight_layout()
        plt.show()

# Group summaries by gender
if 'gender' in df.columns:
    print("\nGroup means by gender:")
    display(df.groupby('gender')[score_cols].mean().round(2))
    print("Group std by gender:")
    display(df.groupby('gender')[score_cols].std().round(2))

# Boxplots by gender
if 'gender' in df.columns:
    for col in score_cols:
        if np.issubdtype(df[col].dtype, np.number):
            plt.figure(figsize=(5,4))
            # build data per group
            data = []
            labels = []
            for g, sub in df.groupby('gender'):
                data.append(sub[col].dropna().values)
                labels.append(str(g))
            plt.boxplot(data, labels=labels, showmeans=True)
            plt.title(f'{col.title()} by Gender')
            plt.xlabel('Gender')
            plt.ylabel(f'{col.title()} Score')
            plt.tight_layout()
            plt.show()
else:
    print("Column 'gender' not found; skipping gender-based plots.")



## 3) Independent Two-Sample t-tests (Male vs Female)
For each score (Math, Reading, Writing):
- Check **normality** per group (Shapiro-Wilk)  
- Check **equal variances** (Levene)  
- Run t-test (Welch by default for safety; standard equal-var t-test if Levene non-significant)  
- Interpret with α = 0.05


In [ ]:

alpha = 0.05

def normality_report(series):
    # Shapiro is sensitive on large N; with N>500, p-values may be tiny even for mild deviations.
    # We'll still report it.
    series = pd.Series(series).dropna()
    # Shapiro max n=5000 in SciPy; subsample if too large
    x = series.sample(min(len(series), 5000), random_state=42)
    stat, p = stats.shapiro(x)
    return stat, p, len(x)

def ttest_gender(df, score_col):
    if 'gender' not in df.columns:
        print(f"[{score_col}] gender column not found; skipping.")
        return

    groups = df['gender'].dropna().unique()
    if len(groups) < 2:
        print(f"[{score_col}] Less than 2 gender groups; skipping.")
        return

    # Assuming 'female'/'male' or similar
    gvals = {}
    for g in groups:
        gvals[g] = df.loc[df['gender']==g, score_col].dropna().values

    # Take first two unique groups deterministically
    g1, g2 = sorted(list(groups))[:2]
    x, y = gvals[g1], gvals[g2]

    # Normality per group
    s1, p1, n1 = normality_report(x)
    s2, p2, n2 = normality_report(y)
    print(f"\n[{score_col.upper()}] Shapiro normality:")
    print(f"  {g1}: n={n1}, stat={s1:.3f}, p={p1:.4g}")
    print(f"  {g2}: n={n2}, stat={s2:.3f}, p={p2:.4g}")

    # Homoscedasticity
    lev_stat, lev_p = stats.levene(x, y, center='median')
    print(f"Levene test (equal variances?): stat={lev_stat:.3f}, p={lev_p:.4g}")

    equal_var = (lev_p >= alpha)
    t_stat, p_val = stats.ttest_ind(x, y, equal_var=equal_var)
    print(f"T-test ({'pooled var' if equal_var else 'Welch'}): t={t_stat:.3f}, p={p_val:.4g}")

    if p_val < alpha:
        print("  -> Reject H0: mean difference is statistically significant.")
    else:
        print("  -> Fail to reject H0: no significant mean difference detected.")

    # Effect size: Cohen's d (pooled if equal_var else Hedges g approx with Welch)
    mean_diff = np.mean(x) - np.mean(y)
    if equal_var:
        sx = np.var(x, ddof=1); sy = np.var(y, ddof=1)
        sp = np.sqrt(((len(x)-1)*sx + (len(y)-1)*sy) / (len(x)+len(y)-2))
        d = mean_diff / (sp + 1e-12)
    else:
        # Use weighted SD approximation
        sx = np.var(x, ddof=1); sy = np.var(y, ddof=1)
        sw = np.sqrt((sx + sy) / 2.0)
        d = mean_diff / (sw + 1e-12)
    print(f"Effect size (approx Cohen's d): {d:.3f}")

for c in score_cols:
    ttest_gender(df, c)



## 4) One-Way ANOVA
Compare **means across multiple groups** for selected factors:
- Example factors: `race/ethnicity`, `lunch`, `test_preparation_course`
- We'll run ANOVA for each available factor on each score, then do **post-hoc** (Tukey HSD if available).


In [ ]:

# Identify candidate multi-level categorical factors
cand_factors = []
for c in ['race/ethnicity','race_ethnicity','race','lunch','test_preparation_course','parental_level_of_education']:
    if c in df.columns:
        cand_factors.append(c)

print('Candidate factors found:', cand_factors)

# Try import Tukey
use_tukey = True
try:
    from statsmodels.stats.multicomp import pairwise_tukeyhsd
    import statsmodels.api as sm
    from statsmodels.formula.api import ols
except Exception as e:
    print("statsmodels not available or import failed; falling back to Bonferroni pairwise t-tests.")
    use_tukey = False

def run_anova_and_posthoc(df, factor, score):
    print(f"\n=== ANOVA for {score.title()} by {factor} ===")
    if df[factor].dropna().nunique() < 2:
        print("Not enough groups for ANOVA.")
        return

    # Drop NA
    sub = df[[factor, score]].dropna()

    if use_tukey:
        # OLS + ANOVA
        model = ols(f'{score} ~ C({factor})', data=sub).fit()
        anova_table = sm.stats.anova_lm(model, typ=2)
        display(anova_table)
        # Tukey HSD
        tuk = pairwise_tukeyhsd(endog=sub[score], groups=sub[factor], alpha=0.05)
        print(tuk)
    else:
        # SciPy one-way ANOVA
        groups = [g[score].dropna().values for _, g in sub.groupby(factor)]
        f_stat, p_val = stats.f_oneway(*groups)
        print(f"One-way ANOVA (SciPy): F={f_stat:.3f}, p={p_val:.4g}")
        # Bonferroni pairwise t-tests
        levels = list(sub[factor].dropna().unique())
        m = len(levels)
        comps = []
        for i in range(m):
            for j in range(i+1, m):
                a = sub.loc[sub[factor]==levels[i], score].values
                b = sub.loc[sub[factor]==levels[j], score].values
                # Welch for safety
                t, p = stats.ttest_ind(a, b, equal_var=False)
                comps.append((levels[i], levels[j], t, p))
        # Bonferroni correction
        k = len(comps)
        print("\nBonferroni-adjusted pairwise comparisons:")
        for (l1, l2, t, p) in comps:
            p_adj = min(1.0, p*k)
            sig = "SIG" if p_adj < 0.05 else "ns"
            print(f"{l1} vs {l2}: t={t:.3f}, p={p:.4g} -> p_adj={p_adj:.4g} [{sig}]")

    # Simple boxplot for visualization
    plt.figure(figsize=(6,4))
    data = [g[score].dropna().values for _, g in sub.groupby(factor)]
    labels = [str(lv) for lv in sub.groupby(factor).groups.keys()]
    plt.boxplot(data, labels=labels, showmeans=True)
    plt.title(f'{score.title()} by {factor}')
    plt.xlabel(factor)
    plt.ylabel(score.title())
    plt.tight_layout()
    plt.show()

# Run ANOVA for available factors on each score
for factor in cand_factors:
    for score in score_cols:
        run_anova_and_posthoc(df, factor, score)



## 5) Insights & Conclusions (Business & Education Context)

- **Gender differences (t-tests):** Summarize which subjects show significant mean differences and the direction (e.g., if `p < 0.05`, indicate which group tends to score higher and by how much on average). Consider **effect sizes** (Cohen’s d) to judge **practical** impact, not just statistical significance.
- **Assumptions:** If normality is violated but sample sizes are large (N > ~30 per group), the t-test is relatively robust. If Levene suggests unequal variances, **Welch’s t-test** is appropriate (we already use this by default).
- **ANOVA findings:** Identify factors (e.g., `race/ethnicity`, `lunch`, `test preparation`) that have significant impacts on scores. Post-hoc tests show *which* groups differ.
- **Recommendations:** For education policy or classroom practice, target support where underperformance is concentrated (e.g., encourage completion of **test preparation**, address disparities linked to **lunch** status).

---

### **XP Ninja — STOP HERE**
Conformément aux consignes, on s’arrête ici pour la partie XP Gold.
